# SQLBuddy 03 - Fine-tuned model evaluation
Same prompts, same greedy decoding (max 256 new tokens), same metrics and the same Spider dev set as
notebook 01 - the only difference is the LoRA adapter from notebook 02 (attached as kernel input),
merged into the base model.

In [ ]:
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 huggingface_hub==2.4.0 nltk==3.9.2

In [ ]:
# Locate our shared code (Kaggle dataset "sqlbuddy-src") and the Spider databases, then import.
import glob, os, shutil, sys, zipfile
work = "/kaggle/working/sqlbuddy"
# The package folder is the one containing run_eval.py + spider_eval/ (Kaggle may unpack it at the
# dataset root or keep a src/ folder); copy it to /kaggle/working/sqlbuddy/src.
hits = [h for h in glob.glob("/kaggle/input/**/run_eval.py", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(h), "spider_eval"))]
if hits:
    shutil.copytree(os.path.dirname(hits[0]), work + "/src", dirs_exist_ok=True)
else:  # or it may still be a zip archive
    zips = glob.glob("/kaggle/input/**/src.zip", recursive=True)
    assert zips, "sqlbuddy-src dataset not attached"
    zipfile.ZipFile(zips[0]).extractall(work + "/src")
sys.path.insert(0, work)
print("src commit:", open(work + "/src/GIT_COMMIT").read().strip() if os.path.exists(work + "/src/GIT_COMMIT") else "?")

from src.data import find_spider_root
SPIDER_ROOT = find_spider_root("/kaggle/input")
print("Spider root:", SPIDER_ROOT)

import torch, transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# The adapter = output folder 'train/adapter' of the 02_train_lora kernel.
ADAPTER = os.path.dirname([p for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
                           if "checkpoint" not in p][0])
print("adapter:", ADAPTER, os.listdir(ADAPTER))

In [ ]:
N_EXAMPLES = 50
from src.run_eval import main
args = ["--out", "/kaggle/working/finetuned", "--adapter", ADAPTER, "--batch-size", "16",
        "--max-new-tokens", "256", "--spider-root", str(SPIDER_ROOT)]
if N_EXAMPLES:
    args += ["--n", str(N_EXAMPLES)]
main(args)

In [ ]:
import pandas as pd
display(pd.read_csv("/kaggle/working/finetuned/metrics.csv"))
pred = pd.read_csv("/kaggle/working/finetuned/predictions.csv")
display(pred[["question", "gold_sql", "pred_sql", "hardness", "exec_match", "exact_match", "valid_sql"]].head(15))